In [0]:
"""
eda_pyspark.py
==============
Reusable EDA utility functions for PySpark DataFrames.
Optimized for Databricks Serverless Compute.

Usage (Databricks notebook):
    %run ./eda_pyspark          # or import after adding to sys.path
    eda = EDA(df)
    eda.overview()
    results = eda.full_report(target="label")
"""

from pyspark.sql import DataFrame, SparkSession
import pyspark.sql.functions as F
import pyspark.sql.types as T
from typing import Optional


class EDA:
    """Core exploratory-data-analysis routines for PySpark DataFrames."""

    # Classify column types once on init
    _NUMERIC_TYPES = (
        T.ByteType, T.ShortType, T.IntegerType, T.LongType,
        T.FloatType, T.DoubleType, T.DecimalType,
    )
    _CATEGORICAL_TYPES = (T.StringType, T.BooleanType)
    _DATETIME_TYPES = (T.DateType, T.TimestampType, T.TimestampNTZType)

    def __init__(self, df: DataFrame) -> None:
        self.df = df.cache()  # cache once — reused across all methods
        self._row_count: Optional[int] = None
        self._numeric_cols: list[str] = [
            f.name for f in df.schema.fields
            if isinstance(f.dataType, self._NUMERIC_TYPES)
        ]
        self._categorical_cols: list[str] = [
            f.name for f in df.schema.fields
            if isinstance(f.dataType, self._CATEGORICAL_TYPES)
        ]
        self._datetime_cols: list[str] = [
            f.name for f in df.schema.fields
            if isinstance(f.dataType, self._DATETIME_TYPES)
        ]

    @property
    def row_count(self) -> int:
        if self._row_count is None:
            self._row_count = self.df.count()
        return self._row_count

    # ------------------------------------------------------------------ #
    # 1. HIGH-LEVEL OVERVIEW
    # ------------------------------------------------------------------ #
    def overview(self) -> DataFrame:
        """Print shape, schema breakdown, and return the schema as a DF."""
        n_rows = self.row_count
        n_cols = len(self.df.columns)
        n_dupes = n_rows - self.df.dropDuplicates().count()

        print(f"Shape       : {n_rows:,} rows × {n_cols} columns")
        print(f"Duplicates  : {n_dupes:,} rows")
        print(f"Numeric     : {len(self._numeric_cols)} cols")
        print(f"Categorical : {len(self._categorical_cols)} cols")
        print(f"Datetime    : {len(self._datetime_cols)} cols")
        print(f"Partitions  : {self.df.rdd.getNumPartitions()}")
        print("\n— Schema ——————————————————————————————")

        schema_rows = [
            (f.name, str(f.dataType), f.nullable)
            for f in self.df.schema.fields
        ]
        spark = SparkSession.getActiveSession()
        schema_df = spark.createDataFrame(schema_rows, ["column", "dtype", "nullable"])
        schema_df.show(n_cols, truncate=False)
        return schema_df

    # ------------------------------------------------------------------ #
    # 2. MISSING VALUES
    # ------------------------------------------------------------------ #
    def missing(self, threshold: float = 0.0) -> DataFrame:
        """Columns with null / NaN counts and percentages.

        Parameters
        ----------
        threshold : float  (0-1)
            Only return columns whose missing ratio exceeds this value.
        """
        n = self.row_count
        exprs = [
            F.sum(
                F.when(F.col(c).isNull() | F.isnan(F.col(c)), 1).otherwise(0)
                if c in self._numeric_cols
                else F.when(F.col(c).isNull(), 1).otherwise(0)
            ).alias(c)
            for c in self.df.columns
        ]
        counts = self.df.select(exprs).collect()[0].asDict()

        spark = SparkSession.getActiveSession()
        rows = [
            (col, cnt, round(cnt / n, 6))
            for col, cnt in counts.items()
            if cnt / n > threshold
        ]
        result = (
            spark.createDataFrame(rows, ["column", "missing_count", "missing_pct"])
            .orderBy(F.col("missing_pct").desc())
        )
        result.show(len(rows), truncate=False)
        return result

    # ------------------------------------------------------------------ #
    # 3. DESCRIPTIVE STATISTICS — NUMERIC
    # ------------------------------------------------------------------ #
    def describe_numeric(self) -> DataFrame:
        """Extended statistics for numeric columns:
        count, mean, stddev, min, 25%, 50%, 75%, max, skewness, kurtosis, IQR, CV.
        """
        cols = self._numeric_cols
        if not cols:
            print("No numeric columns found.")
            return self.df.limit(0)

        agg_exprs = []
        for c in cols:
            col = F.col(c)
            agg_exprs.extend([
                F.count(col).alias(f"{c}__count"),
                F.mean(col).alias(f"{c}__mean"),
                F.stddev(col).alias(f"{c}__stddev"),
                F.min(col).alias(f"{c}__min"),
                F.expr(f"percentile_approx({c}, 0.25)").alias(f"{c}__q1"),
                F.expr(f"percentile_approx({c}, 0.50)").alias(f"{c}__median"),
                F.expr(f"percentile_approx({c}, 0.75)").alias(f"{c}__q3"),
                F.max(col).alias(f"{c}__max"),
                F.skewness(col).alias(f"{c}__skewness"),
                F.kurtosis(col).alias(f"{c}__kurtosis"),
            ])

        stats = self.df.agg(*agg_exprs).collect()[0].asDict()

        spark = SparkSession.getActiveSession()
        rows = []
        for c in cols:
            q1 = stats[f"{c}__q1"]
            q3 = stats[f"{c}__q3"]
            mean = stats[f"{c}__mean"]
            std = stats[f"{c}__stddev"]
            iqr = (q3 - q1) if q1 is not None and q3 is not None else None
            cv = (std / mean) if mean and mean != 0 and std is not None else None
            rows.append((
                c,
                stats[f"{c}__count"],
                _r(mean), _r(std),
                _r(stats[f"{c}__min"]),
                _r(q1), _r(stats[f"{c}__median"]), _r(q3),
                _r(stats[f"{c}__max"]),
                _r(stats[f"{c}__skewness"]),
                _r(stats[f"{c}__kurtosis"]),
                _r(iqr), _r(cv),
            ))

        result = spark.createDataFrame(
            rows,
            ["column", "count", "mean", "stddev", "min",
             "q1", "median", "q3", "max",
             "skewness", "kurtosis", "iqr", "cv"],
        )
        result.show(len(cols), truncate=False)
        return result

    # ------------------------------------------------------------------ #
    # 4. DESCRIPTIVE STATISTICS — CATEGORICAL
    # ------------------------------------------------------------------ #
    def describe_categorical(self) -> DataFrame:
        """Summary for string / boolean columns: unique count, top value, freq, missing %."""
        cols = self._categorical_cols
        if not cols:
            print("No categorical columns found.")
            return self.df.limit(0)

        n = self.row_count
        spark = SparkSession.getActiveSession()
        rows = []

        # Single pass: compute distinct + nulls per column
        agg_exprs = []
        for c in cols:
            agg_exprs.append(F.countDistinct(F.col(c)).alias(f"{c}__nunique"))
            agg_exprs.append(F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(f"{c}__nulls"))
        agg_result = self.df.agg(*agg_exprs).collect()[0].asDict()

        # Top value per column (separate grouped queries — unavoidable)
        for c in cols:
            top = (
                self.df.filter(F.col(c).isNotNull())
                .groupBy(c)
                .count()
                .orderBy(F.col("count").desc())
                .limit(1)
                .collect()
            )
            top_val = top[0][c] if top else None
            top_freq = top[0]["count"] if top else 0
            rows.append((
                c,
                agg_result[f"{c}__nunique"],
                top_val,
                top_freq,
                round(agg_result[f"{c}__nulls"] / n, 6),
            ))

        result = spark.createDataFrame(
            rows, ["column", "n_unique", "top_value", "top_freq", "missing_pct"]
        )
        result.show(len(cols), truncate=False)
        return result

    # ------------------------------------------------------------------ #
    # 5. OUTLIER DETECTION (IQR)
    # ------------------------------------------------------------------ #
    def outliers_iqr(self, factor: float = 1.5) -> DataFrame:
        """Flag outlier counts per numeric column using the IQR method."""
        cols = self._numeric_cols
        if not cols:
            print("No numeric columns.")
            return self.df.limit(0)

        # Compute Q1/Q3 in one pass
        quantile_exprs = []
        for c in cols:
            quantile_exprs.append(F.expr(f"percentile_approx({c}, 0.25)").alias(f"{c}__q1"))
            quantile_exprs.append(F.expr(f"percentile_approx({c}, 0.75)").alias(f"{c}__q3"))
        q_vals = self.df.agg(*quantile_exprs).collect()[0].asDict()

        # Count outliers in one pass
        outlier_exprs = []
        bounds = {}
        for c in cols:
            q1 = q_vals[f"{c}__q1"]
            q3 = q_vals[f"{c}__q3"]
            if q1 is None or q3 is None:
                continue
            iqr = q3 - q1
            lo, hi = q1 - factor * iqr, q3 + factor * iqr
            bounds[c] = (lo, hi)
            outlier_exprs.append(
                F.sum(
                    F.when((F.col(c) < lo) | (F.col(c) > hi), 1).otherwise(0)
                ).alias(f"{c}__outliers")
            )

        if not outlier_exprs:
            return self.df.limit(0)

        o_vals = self.df.agg(*outlier_exprs).collect()[0].asDict()
        n = self.row_count

        spark = SparkSession.getActiveSession()
        rows = []
        for c, (lo, hi) in bounds.items():
            cnt = o_vals[f"{c}__outliers"]
            rows.append((c, _r(lo), _r(hi), cnt, round(cnt / n, 6)))

        result = (
            spark.createDataFrame(rows, ["column", "lower_bound", "upper_bound", "n_outliers", "outlier_pct"])
            .orderBy(F.col("outlier_pct").desc())
        )
        result.show(len(rows), truncate=False)
        return result

    # ------------------------------------------------------------------ #
    # 6. CARDINALITY
    # ------------------------------------------------------------------ #
    def cardinality(self) -> DataFrame:
        """Unique-value counts and uniqueness ratio for every column."""
        n = self.row_count
        exprs = [F.countDistinct(F.col(c)).alias(c) for c in self.df.columns]
        distincts = self.df.agg(*exprs).collect()[0].asDict()

        spark = SparkSession.getActiveSession()
        rows = [
            (
                c,
                str(self.df.schema[c].dataType),
                distincts[c],
                round(distincts[c] / n, 6),
                distincts[c] == n,
            )
            for c in self.df.columns
        ]
        result = (
            spark.createDataFrame(rows, ["column", "dtype", "n_unique", "unique_pct", "is_potential_id"])
            .orderBy(F.col("n_unique").desc())
        )
        result.show(len(rows), truncate=False)
        return result

    # ------------------------------------------------------------------ #
    # 7. CORRELATIONS
    # ------------------------------------------------------------------ #
    def correlation_matrix(self, method: str = "pearson") -> DataFrame:
        """Pairwise correlation for numeric columns.

        Parameters
        ----------
        method : str   'pearson' or 'spearman'
        """
        cols = self._numeric_cols
        if len(cols) < 2:
            print("Need ≥ 2 numeric columns for correlations.")
            return self.df.limit(0)

        spark = SparkSession.getActiveSession()
        rows = []
        for c1 in cols:
            row = {"column": c1}
            for c2 in cols:
                if method == "spearman":
                    # Spearman via rank transform
                    val = _spearman(self.df, c1, c2)
                else:
                    val = self.df.stat.corr(c1, c2)
                row[c2] = _r(val)
            rows.append(row)

        schema_fields = [T.StructField("column", T.StringType())] + [
            T.StructField(c, T.DoubleType()) for c in cols
        ]
        result = spark.createDataFrame(rows, T.StructType(schema_fields))
        result.show(len(cols), truncate=False)
        return result

    def top_correlations(self, n: int = 10, method: str = "pearson") -> DataFrame:
        """Return the top-n strongest pairwise correlations (excluding self)."""
        cols = self._numeric_cols
        spark = SparkSession.getActiveSession()
        pairs = []

        for i, c1 in enumerate(cols):
            for c2 in cols[i + 1:]:
                if method == "spearman":
                    val = _spearman(self.df, c1, c2)
                else:
                    val = self.df.stat.corr(c1, c2)
                if val is not None:
                    pairs.append((c1, c2, round(val, 6), round(abs(val), 6)))

        result = (
            spark.createDataFrame(pairs, ["var_1", "var_2", "corr", "abs_corr"])
            .orderBy(F.col("abs_corr").desc())
            .limit(n)
        )
        result.show(n, truncate=False)
        return result

    # ------------------------------------------------------------------ #
    # 8. VALUE-FREQUENCY TABLE
    # ------------------------------------------------------------------ #
    def value_counts(self, col: str, top_n: int = 10) -> DataFrame:
        """Frequency table with cumulative percentages for a single column."""
        from pyspark.sql.window import Window

        n = self.row_count
        result = (
            self.df.groupBy(col)
            .count()
            .orderBy(F.col("count").desc())
            .limit(top_n)
            .withColumn("pct", F.round(F.col("count") / F.lit(n), 6))
            .withColumn(
                "cum_pct",
                F.round(
                    F.sum("pct").over(Window.orderBy(F.col("count").desc()).rowsBetween(
                        Window.unboundedPreceding, Window.currentRow
                    )),
                    6,
                ),
            )
        )
        result.show(top_n, truncate=False)
        return result

    # ------------------------------------------------------------------ #
    # 9. TARGET ANALYSIS
    # ------------------------------------------------------------------ #
    def target_summary(self, target: str) -> DataFrame:
        """Quick summary of a target variable (classification vs regression)."""
        n_unique = self.df.select(F.countDistinct(target)).collect()[0][0]

        if n_unique <= 20 or target in self._categorical_cols:
            print(f"— Classification target: '{target}' ({n_unique} classes) ————")
            n = self.row_count
            dist = (
                self.df.groupBy(target)
                .count()
                .withColumn("pct", F.round(F.col("count") / F.lit(n), 6))
                .orderBy(F.col("count").desc())
            )
            dist.show(n_unique, truncate=False)
            pcts = [row["pct"] for row in dist.collect()]
            print(f"Balance ratio (min/max class): {min(pcts) / max(pcts):.4f}")
            return dist
        else:
            print(f"— Regression target: '{target}' ——————————————————")
            stats = self.df.select(
                F.count(target).alias("count"),
                F.mean(target).alias("mean"),
                F.stddev(target).alias("stddev"),
                F.min(target).alias("min"),
                F.expr(f"percentile_approx({target}, 0.5)").alias("median"),
                F.max(target).alias("max"),
                F.skewness(target).alias("skewness"),
                F.kurtosis(target).alias("kurtosis"),
            )
            stats.show(truncate=False)
            return stats

    # ------------------------------------------------------------------ #
    # 10. FULL REPORT
    # ------------------------------------------------------------------ #
    def full_report(self, target: Optional[str] = None) -> dict:
        """Run all EDA steps and return results as a dict of DataFrames."""
        print("=" * 65)
        print("           EXPLORATORY DATA ANALYSIS REPORT (PySpark)")
        print("=" * 65)

        results = {}

        print("\n" + "=" * 65)
        print("  1. OVERVIEW")
        print("=" * 65)
        results["schema"] = self.overview()

        print("\n" + "=" * 65)
        print("  2. MISSING VALUES")
        print("=" * 65)
        results["missing"] = self.missing()

        print("\n" + "=" * 65)
        print("  3. NUMERIC STATISTICS")
        print("=" * 65)
        results["numeric_stats"] = self.describe_numeric()

        print("\n" + "=" * 65)
        print("  4. CATEGORICAL STATISTICS")
        print("=" * 65)
        results["categorical_stats"] = self.describe_categorical()

        print("\n" + "=" * 65)
        print("  5. OUTLIERS (IQR)")
        print("=" * 65)
        results["outliers"] = self.outliers_iqr()

        print("\n" + "=" * 65)
        print("  6. CARDINALITY")
        print("=" * 65)
        results["cardinality"] = self.cardinality()

        print("\n" + "=" * 65)
        print("  7. TOP CORRELATIONS")
        print("=" * 65)
        results["top_correlations"] = self.top_correlations()

        if target and target in self.df.columns:
            print("\n" + "=" * 65)
            print(f"  8. TARGET: '{target}'")
            print("=" * 65)
            results["target"] = self.target_summary(target)

        print("\n" + "=" * 65)
        print("  REPORT COMPLETE")
        print("=" * 65)
        return results

    def unpersist(self) -> None:
        """Release the cached DataFrame when done."""
        self.df.unpersist()


# ------------------------------------------------------------------ #
# MODULE-LEVEL HELPERS
# ------------------------------------------------------------------ #
def _r(val, decimals: int = 6):
    """Round a value safely (handles None)."""
    if val is None:
        return None
    return round(float(val), decimals)


def _spearman(df: DataFrame, c1: str, c2: str) -> Optional[float]:
    """Compute Spearman correlation via rank transform."""
    from pyspark.sql.window import Window

    ranked = (
        df.select(c1, c2)
        .dropna(subset=[c1, c2])
        .withColumn(f"_rank_{c1}", F.percent_rank().over(Window.orderBy(c1)))
        .withColumn(f"_rank_{c2}", F.percent_rank().over(Window.orderBy(c2)))
    )
    return ranked.stat.corr(f"_rank_{c1}", f"_rank_{c2}")